<a href="https://colab.research.google.com/github/nehalnady/DM_Project/blob/main/DM_Regression_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.feature_selection import SelectKBest, f_regression


df = pd.read_csv('/content/vehicles.csv', engine='python', on_bad_lines='warn')
print(df.head())
print("cloumns: ", df.columns)
df.columns = df.columns.str.strip().str.lower()

           id                                                url  \
0  7222695916  https://prescott.craigslist.org/cto/d/prescott...   
1  7218891961  https://fayar.craigslist.org/ctd/d/bentonville...   
2  7221797935  https://keys.craigslist.org/cto/d/summerland-k...   
3  7222270760  https://worcester.craigslist.org/cto/d/west-br...   
4  7210384030  https://greensboro.craigslist.org/cto/d/trinit...   

                   region                         region_url  price  year  \
0                prescott    https://prescott.craigslist.org   6000   NaN   
1            fayetteville       https://fayar.craigslist.org  11900   NaN   
2            florida keys        https://keys.craigslist.org  21000   NaN   
3  worcester / central MA   https://worcester.craigslist.org   1500   NaN   
4              greensboro  https://greensboro.craigslist.org   4900   NaN   

  manufacturer model condition cylinders  ... size  type paint_color  \
0          NaN   NaN       NaN       NaN  ...  NaN   NaN

In [14]:


# # Clean column names
# df.columns = df.columns.str.strip().str.lower()

# print("Columns:", df.columns.tolist())

# # =========================
# # 1. Remove duplicates
# # =========================
# df.drop_duplicates(inplace=True)

# # =========================
# # 2. Drop useless columns EARLY
# # =========================
# drop_cols = [
#     "id", "url", "region_url", "image_url",
#     "description", "vin", "posting_date"
# ]
# df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

# # =========================
# # 3. Handle numeric columns
# # =========================
# numeric_cols = ["year", "odometer", "price", "lat", "long"]

# for col in numeric_cols:
#     if col in df.columns:
#         df[col] = pd.to_numeric(df[col], errors='coerce')
#         df[col].fillna(df[col].median(), inplace=True)

# # =========================
# # 4. Outlier capping
# # =========================
# for col in ["price", "odometer", "year"]:
#     if col in df.columns:
#         Q1 = df[col].quantile(0.25)
#         Q3 = df[col].quantile(0.75)
#         IQR = Q3 - Q1
#         df[col] = df[col].clip(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

# # =========================
# # 5. Drop high-cardinality columns
# # =========================
# high_card_cols = ["model", "region"]
# df.drop(columns=[c for c in high_card_cols if c in df.columns], inplace=True)

# # =========================
# # 6. One-Hot Encoding ONLY
# # =========================
# df = pd.get_dummies(df, drop_first=True)

# print("Shape after encoding:", df.shape)

# # =========================
# # 7. FINAL NaN cleanup (CRITICAL ✅)
# # =========================
# df.fillna(df.median(numeric_only=True), inplace=True)

# # =========================
# # 8. Feature / target split
# # =========================
# X = df.drop("price", axis=1)
# y = df["price"]

# # =========================
# # 9. Scaling
# # =========================
# scaler = StandardScaler()
# X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# # Safety check (IMPORTANT)
# print("Remaining NaNs:", X_scaled.isnull().sum().sum())

# # =========================
# # 10. Feature selection
# # =========================
# selector = SelectKBest(score_func=f_regression, k=min(10, X.shape[1]))
# X_selected = selector.fit_transform(X_scaled, y)

# selected_features = X.columns[selector.get_support()]
# print("\nSelected features:", list(selected_features))

Columns: ['price', 'year', 'manufacturer', 'condition', 'fuel', 'odometer', 'title_status', 'transmission', 'drive', 'type', 'paint_color', 'county', 'state', 'lat', 'long', 'cylinders_12 cylinders', 'cylinders_3 cylinders', 'cylinders_4 cylinders', 'cylinders_5 cylinders', 'cylinders_6 cylinders', 'cylinders_8 cylinders', 'cylinders_other', 'size_full-size', 'size_mid-size', 'size_sub-compact']
Shape after encoding: (128351, 25)
Remaining NaNs: 128351


ValueError: Input X contains NaN.
SelectKBest does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
import warnings
warnings.filterwarnings("ignore") # Ignore warnings

# ── B1. Remove duplicates ────────────────────────────────────
before = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"  [1] Duplicates removed : {before - len(df)}  →  {len(df)} rows remain")

# ── B2. Drop irrelevant or high-cardinality columns ───────────
# Adapt drop_cols for the 'vehicles.csv' dataset.
# Using columns identified as useless or high-cardinality in the original context (e.g., cell IKkTHKnvs0_L)
drop_cols = [
    "id", "url", "region_url", "image_url", "description", "vin", "posting_date", # From previous (commented) cell's approach
    "county", # This column is almost entirely NaN based on `missing` variable in kernel state
    "model", "region" # High cardinality, from previous (commented) cell's approach
]
# Ensure only existing columns are dropped
df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True)
print(f"  [2] Dropped columns: {', '.join([col for col in drop_cols if col in df.columns])}")

# ── B3. Handle numeric columns: type conversion and initial NaN imputation ──────────────────────
numeric_cols = ["year", "odometer", "price", "lat", "long"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        # Impute NaNs with median for numeric columns
        if df[col].isnull().any():
            df[col].fillna(df[col].median(), inplace=True)
print("  [3] Processed numeric columns (type converted & NaNs filled with median):", ", ".join([c for c in numeric_cols if c in df.columns]))

# ── B4. Outlier detection & capping (IQR) ───────────────────────────
outlier_targets = ["price", "odometer", "year"] # Use actual numeric columns
print(f"\n  [4] Outlier detection (IQR):")
for col in outlier_targets:
    if col in df.columns:
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        n_out = ((df[col] < lo) | (df[col] > hi)).sum()
        df[col] = df[col].clip(lo, hi)
        print(f"    {col:20s} : {n_out:4d} outliers capped")

# ── B5. Feature engineering: car age ────────────────────────
current_year = 2024
if "year" in df.columns:
    df["car_age"] = current_year - df["year"]
    df.drop(columns=["year"], inplace=True)
    print(f"\n  [5] Created 'car_age' from 'year'")
else:
    print(f"\n  [5] 'year' column not found, 'car_age' not created.")

# ── B6. Handle missing values (categorical) and Label Encode ────────────────────────────────
# Identify all remaining categorical columns (object type)
print(f"\n  [6] Missing values before categorical imputation and encoding:")
missing_before_cat_impute = df.isnull().sum()
if missing_before_cat_impute[missing_before_cat_impute > 0].empty:
    print("    No missing categorical values.")
else:
    print(missing_before_cat_impute[missing_before_cat_impute > 0].to_string())

le = LabelEncoder()
categorical_cols_to_encode = df.select_dtypes(include='object').columns.tolist()

for col in categorical_cols_to_encode:
    # Fill NaNs with mode before encoding
    if df[col].isnull().any():
        df[col].fillna(df[col].mode()[0], inplace=True)
    df[col] = le.fit_transform(df[col].astype(str))
print("  → Categorical NaNs filled with mode & Label-encoded :", ', '.join(categorical_cols_to_encode))
print(f"  Shape after encoding : {df.shape}")

# ── B7. Feature / target split ──────────────────────────────
target_col = "price" # 'selling_price' from original cell is 'price' in 'vehicles.csv'

if target_col in df.columns:
    X = df.drop(columns=[target_col])
    y = df[target_col]
    print(f"\n  [7] Feature/target split: X (features), y (target='{target_col}')")
else:
    X = pd.DataFrame() # Initialize empty in case target not found
    y = pd.Series()
    print(f"\n  [7] Target column '{target_col}' not found. Cannot perform feature/target split.")

# ── B8. Feature scaling ─────────────────────────────────────
# Final safety fill — catch any NaNs that survived, especially important for X before scaling
if not X.empty:
    if X.isnull().sum().sum() > 0:
        print("\n  [8] Warning: NaNs found in X before scaling. Imputing with median.")
        X.fillna(X.median(numeric_only=True), inplace=True)

    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
    print("  [8] Features scaled using StandardScaler.")
else:
    X_scaled = pd.DataFrame()
    print("  [8] Feature scaling skipped as X is empty.")

# ── B9. Feature selection (SelectKBest, top-10 or all) ──────
if not X_scaled.empty and not y.empty:
    k = min(10, X_scaled.shape[1])
    selector = SelectKBest(score_func=f_regression, k=k)

    # Re-check for NaNs before fit_transform (defense-in-depth)
    if X_scaled.isnull().sum().sum() > 0:
        print("\n  [9] Warning: NaNs found in X_scaled before feature selection. Imputing with median.")
        X_scaled.fillna(X_scaled.median(numeric_only=True), inplace=True)

    X_selected = selector.fit_transform(X_scaled, y)
    selected_features = X.columns[selector.get_support()]

    print(f"\n  [9] Feature selection — kept {k} / {X_scaled.shape[1]} features")
    print(f"   Selected : {list(selected_features)}")
else:
    print("\n  [9] Feature selection skipped due to empty X_scaled or y.")


  [1] Duplicates removed : 0  →  128351 rows remain
  [2] Dropped columns: 
  [3] Processed numeric columns (type converted & NaNs filled with median): year, odometer, price, lat, long

  [4] Outlier detection (IQR):
    price                :    0 outliers capped
    odometer             :    0 outliers capped
    year                 :    0 outliers capped

  [5] Created 'car_age' from 'year'

  [6] Missing values before categorical imputation and encoding:
    No missing categorical values.
  → Categorical NaNs filled with mode & Label-encoded : 
  Shape after encoding : (128351, 24)

  [7] Feature/target split: X (features), y (target='price')
  [8] Features scaled using StandardScaler.

  [9] Feature selection — kept 10 / 23 features
   Selected : ['odometer', 'title_status', 'transmission', 'type', 'paint_color', 'cylinders_4 cylinders', 'cylinders_8 cylinders', 'size_full-size', 'size_mid-size', 'car_age']
